In [1]:
import os
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import pandas as pd
from torch.utils.data import DataLoader, Dataset, random_split
import GPUtil
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from torch.optim import AdamW,SGD,Adam
import json
import numpy as np

In [19]:
train_path = '../data/translation2019zh/translation2019zh_train.json'
# use 50 * 200 rows to train
chunk_size = 50
chunk_num = 256


In [20]:
df = pd.DataFrame()
with pd.read_json(train_path, lines=True, chunksize=chunk_size) as reader:
    count = 0
    while count < chunk_num:
        chunk = next(iter(reader))
        df = pd.concat([df, chunk], axis=0)
        count += 1


In [44]:
sample_idx = 0
en, zh = df.iloc[sample_idx, :].values
tokenizer_en = BertTokenizer.from_pretrained('bert-base-cased')
tokenizer_zh = BertTokenizer.from_pretrained('bert-base-chinese')
tokens_en = tokenizer_en.tokenize(en)
ids_en = tokenizer_en.convert_tokens_to_ids(tokens_en)
print(ids_en)
print(type(ids_en))

[1370, 3407, 4295, 1757, 117, 1133, 1114, 170, 6812, 2773, 1107, 9478, 8405, 117, 1128, 1169, 1329, 170, 122, 131, 122, 4267, 18404, 1104, 1142, 9991, 119]
<class 'list'>


In [58]:
df.head()

,english,chinese
0,"For greater sharpness, but with a slight incre...",为了更好的锐度，但是附带的会多一些颗粒度，可以使用这个显影剂的1：1稀释液。
1,"He calls the Green Book, his book of teachings...",他还把宣扬自己思想的所谓《绿皮书》称作“新福音书”。
2,And the light breeze moves me to caress her lo...,微风推着我去爱抚它的长耳朵
3,They have the blood of martyrs is the White to...,它们的先烈们的鲜血是白流了…
4,"Finally, the Lakers head to the Motor City to ...",最后，在1月31日，湖人将前往汽车城底特律挑战活塞队，活塞近来在东部排名第二。


In [24]:
MAX_DATASET_SIZE = 330000
TRAIN_SIZE = 300000
VALID_SIZE = 30000

In [25]:


class TranslationDataSet(Dataset):
    def __init__(self, file_path):
        self.data = self.load_data(file_path)

    def load_data(self, file_path):
        Data = {}
        with open(file_path, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= MAX_DATASET_SIZE:
                    break
                sample = json.loads(line.strip())
                Data[idx] = sample
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

path = '../data/translation2019zh/translation2019zh_train.json'
test_path = '../data/translation2019zh/translation2019zh_valid.json'
data = TranslationDataSet(path)
train_data, valid_data = random_split(data, [TRAIN_SIZE, VALID_SIZE])
test_data = TranslationDataSet(test_path)

In [26]:
train_data[1]

{'english': 'Jennifer : Let me see that Palm gadget.',
 'chinese': '珍妮花 ：让我瞧瞧这掌上型的小玩意儿。'}

In [27]:
model_checkpoint = "Helsinki-NLP/opus-mt-zh-en"
# model_checkpoint ="moxying/opus-mt-zh-en"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

MAX_INPUT_SIZE = 128
MAX_OUTPUT_SIZE = 128
BATCH_SIZE = 16

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using' + device)

inputs = [train_data[i]["chinese"] for i in range(BATCH_SIZE)]
targets = [train_data[i]["english"] for i in range(BATCH_SIZE)]

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

def collate(batch_samples):
    batch_inputs, batch_targets = [], []
    for sample in batch_samples:
        batch_inputs.append(sample['chinese'])
        batch_targets.append(sample['english'])

    batch_data = tokenizer(
        batch_inputs,
        padding=True,
        max_length=MAX_INPUT_SIZE,
        truncation=True,
        return_tensors="pt"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch_targets,
            padding=True,
            max_length=MAX_OUTPUT_SIZE,
            truncation=True,
            return_tensors="pt"
        )["input_ids"]

        batch_data['decoder_input_ids'] = model.prepare_decoder_input_ids_from_labels(labels)
        end_token_index = torch.where(labels == tokenizer.eos_token_id)[1]
        for idx, end_idx in enumerate(end_token_index):
            labels[idx][end_idx+1: ] = -100
        batch_data['labels'] = labels
    return batch_data

train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate)
valid_dataloader = DataLoader(valid_data, batch_size=32, shuffle=False, collate_fn=collate)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate)
# with tokenizer.as_target_tokenizer():
#     labels = tokenizer(targets, padding=True, max_length=MAX_INPUT_SIZE, truncation=True, return_tensors="pt")["input_ids"]

# end_index = torch.where(labels==tokenizer.eos_token_id)[1]
# for idx, end_idx in enumerate(end_index):
#     labels[idx][end_idx+1] = -100

usingcuda


In [28]:
train_dataloader

In [39]:
model_inputs = tokenizer(
    inputs, 
    padding=True, 
    max_length=MAX_INPUT_SIZE, 
    truncation=True,
    return_tensors="pt"
)
print(inputs)

['在一定范围内适量施用氮磷可以使北美海蓬子的总鲜物量和菜用产量显著增加。', '珍妮花 ：让我瞧瞧这掌上型的小玩意儿。', '他的写作质量随着他的酗喝而变质…', '所以不该采用人格理论了。', '影视详情： 6个月的车新婚的工会和优美。', '问下那些获救的智利矿工就知道提高铜矿的生产率有多难。', '汤亭亭的小说《女勇士》是当代美国华裔英语文学的代表作品之一。', '她是1985年普林斯顿大学的优秀毕业生。 那时候，她和另外三个女人同居一室，去洗手间还得下三层楼梯。', '要想理解创作共用在这方面是如何创新的，我们可以回顾一下 会议页面，来看看我们是如何在页脚标识版本信息的。', '验血结果（不仅能够告诉人们能够活多久）还会受这类公司的关注，如需要制定人寿保险单条款的保险公司和受突然重病或者过早死亡等风险因素影响的医疗公司。', '在上述分析的基础上，对本区土地进行了类型划分，建立起二级土地类型自然分类系统；', '虽然可以仅仅执行安装，在安装前后不进行任何配置，但是我们建议目标系统至少有 1GB 的空闲 RAM 和 1GB 的空闲磁盘空间。', '这匹配具有 ref 属性的所有元素。', '对水基液压液生物降解性的评价方法和试验方法的选择进行了归纳和总结；', '大多数人都知道“山姆大叔”指美国政府，但是设想我们不了解这个词，我们就会弄不清是什么意思。', '所以，OmniFind 连接器只为内容支持一种单一的全局项目类，这样每个项目可能会有无限多的专门的其他属性。']


In [30]:
from tqdm.auto import tqdm

def train_step(dataloader, model, optimzer, lr_scheduler, epoch, total_loss):
    # bar = tqdm(range(len(dataloader)))
    # bar.set_description(f'loss:{0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)

    model.train()
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        outputs = model(**batch_data)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        # bar.set_description(f'loss: {total_loss/(finish_batch_num + batch):>7f}')
        # bar.update(1)

    return total_loss

    

In [31]:
from sacrebleu.metrics import BLEU
bleu = BLEU()

def test_step(dataloader, model):
    preds, labels = [], []

    model.eval()
    for batch_data in tqdm(dataloader):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=MAX_OUTPUT_SIZE,
            ).cpu().numpy()
        label_tokens = batch_data["labels"].cpu().numpy()

        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        label_tokens = np.where(label_tokens!=-100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        preds += [pred.strip() for pred in decoded_preds]
        labels += [[label.strip()] for label in decoded_labels]
    bleu_score = bleu.corpus_score(preds, labels).score
    print(f'BLEU: {bleu_score:>0.2f}\n')
    return bleu_score

In [32]:
from transformers import AdamW, get_scheduler

learning_rate = 2e-5
EPOCHS = 3

optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=EPOCHS*len(train_dataloader)
)

total_loss = 0.
best_bleu = 0.
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}\n---------------------------")
    total_loss = train_step(train_dataloader, model, optimizer, lr_scheduler, epoch+1, total_loss)
    valid_bleu = test_step(valid_dataloader, model)
    if valid_bleu > best_bleu:
        best_bleu = valid_bleu
        print("saving new W\n")
        torch.save(
            model.state_dict(),
            f'epoch_{epoch+1}_valid_bleu_{valid_bleu:0.2f}_model_weights.bin'
        )
print('!123123')

E:\codeAndProject\DataRepairEnv\Lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
E:\codeAndProject\DataRepairEnv\Lib\site-packages\transformers\tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Epoch 1
---------------------------


  0%|          | 0/938 [00:00<?, ?it/s]

BLEU: 45.20

saving new W

Epoch 2
---------------------------


  0%|          | 0/938 [00:00<?, ?it/s]

BLEU: 45.20

Epoch 3
---------------------------


  0%|          | 0/938 [00:00<?, ?it/s]

BLEU: 46.01

saving new W

!123123


In [33]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.load_state_dict(torch.load('epoch_3_valid_bleu_46.01_model_weights.bin'))

C:\Users\OZCQC\AppData\Local\Temp\ipykernel_30036\3689557768.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('epoch_3_valid_bleu_46.01_m

<All keys matched successfully>

In [35]:
test = "你为什么是复读机，你能不能不要当复读机。"
test_tokens = tokenizer.tokenize(test)

print(test_tokens)
ids = tokenizer.convert_tokens_to_ids(test_tokens)
print(ids)
ids_tensors = torch.tensor([ids])
print(ids_tensors.device)
ids_tensors = ids_tensors
print(ids_tensors.device)
print(next(model.parameters()).device)
# outputs = model.generate(ids_tensors)
generated = model.generate(
                ids_tensors,
                max_length=MAX_OUTPUT_SIZE,
            ).cpu().numpy()

['▁你为什么', '是', '复', '读', '机', ',', '你', '能不能', '不要', '当', '复', '读', '机', '。']
[20926, 69, 7597, 6970, 2326, 2, 146, 16918, 2958, 1891, 7597, 6970, 2326, 9]
cpu
cpu
cpu


In [36]:
print(generated.shape)
a = tokenizer.batch_decode(generated, skip_special_tokens=True)
print(a)

(1, 32)
["Why why are you a repeat reader? Why are you a repeat reader? Would you mind if you wouldn't mind not being a repeat reader?"]


In [37]:
model.load_state_dict(torch.load('epoch_1_valid_bleu_52.81_model_weights.bin'))
with torch.no_grad():
    print('evaluating')
    sources, preds, labels = [], [], []
    for batch_data in tqdm(test_dataloader):
        batch_data = batch_data.to(device)
        generated_tokens = model.generate(
            batch_data["input_ids"],
            attention_mask=batch_data["attention_mask"],
            max_length=MAX_OUTPUT_SIZE,
        ).cpu().numpy()
        label_tokens = batch_data["labels"].cpu().numpy()

        decoded_sources = tokenizer.batch_decode(
            batch_data["input_ids"].cpu().numpy(),
            skip_special_tokens=True,
            use_source_tokenizer=True
        )
        
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        label_tokens = np.where(label_tokens!=-100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        sources += [source.strip() for source in decoded_sources]
        preds += [pred.strip() for pred in decoded_preds]
        labels += [[label.strip()] for label in decoded_labels]

    bleu_score = bleu.corpus_score(preds, labels).score
    print(bleu_score)


C:\Users\OZCQC\AppData\Local\Temp\ipykernel_30036\674897149.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('epoch_1_valid_bleu_52.81_mo

evaluating


  0%|          | 0/1229 [00:00<?, ?it/s]

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [38]:
torch.cuda.empty_cache()